# Chapter 6 Lab — Simulating Coupling Dynamics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/liquid-books/basal-cognition/blob/main/notebooks/ch06-lab-coupling-sim.ipynb)

**Basal Cognition · Dr. Ernesto Lee**

---

## What you will build

A simulation of ten agents sharing state through a tunable coupling weight:

- Each **agent** holds a value that drifts randomly over time (noise).
- A **coupling parameter** controls how much each agent pulls toward the group average.
- At **zero coupling**, agents drift apart — a colony.
- At **full coupling**, one corrupted agent drags everyone — an arrhythmia.
- You will find the coupling level that **maximizes coherence while containing error propagation**.

This is the gap junction experiment in code. Same agents, different coupling — different individual.

**Estimated time:** 30–45 minutes

**No prior Python experience needed.** Every block is commented.

In [ ]:
# Install dependencies (only needed if running outside Colab)
# In Colab, matplotlib and numpy are already available.
%pip install -q matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Set a random seed so results are reproducible
np.random.seed(42)

print('Imports complete. Ready to simulate coupling.')

## Step 1 — Define the parameters

These control the simulation. You will change `COUPLING` to explore different regimes.

In [ ]:
# Number of agents (cells in the tissue)
N_AGENTS = 10

# Number of time steps to simulate
N_STEPS = 200

# Noise level — how much each agent drifts randomly per step
NOISE = 0.3

# Coupling weight — how strongly each agent pulls toward the group average
# 0.0 = no coupling (colony), 1.0 = full coupling (over-coupled)
# START HERE: try 0.0, then 0.3, then 0.8, then 1.0
COUPLING = 0.3

# Corruption step — at this timestep, one agent gets a large bad value
CORRUPTION_STEP = 100
CORRUPTION_MAGNITUDE = 5.0  # how bad the corrupted signal is
CORRUPTED_AGENT = 0          # which agent gets corrupted (0 = first agent)

print(f'Parameters set. Coupling = {COUPLING}, Noise = {NOISE}')
print(f'Corruption injected at step {CORRUPTION_STEP} into agent {CORRUPTED_AGENT}')

## Step 2 — Run the simulation

Each step:
1. Each agent drifts by a random amount (noise).
2. Each agent pulls toward the group average by `COUPLING` fraction.
3. At `CORRUPTION_STEP`, one agent gets a large jolt.

We record the full history to plot later.

In [ ]:
def run_simulation(n_agents, n_steps, noise, coupling, corruption_step, corruption_magnitude, corrupted_agent):
    """Run the coupling simulation. Returns history array of shape (n_steps, n_agents)."""
    # Initialize all agents at value 0
    state = np.zeros(n_agents)
    history = np.zeros((n_steps, n_agents))

    for t in range(n_steps):
        # Add random drift (noise)
        state = state + np.random.randn(n_agents) * noise

        # Inject corruption at the designated step
        if t == corruption_step:
            state[corrupted_agent] += corruption_magnitude
            print(f'  [Step {t}] Corruption injected into agent {corrupted_agent}: value = {state[corrupted_agent]:.2f}')

        # Pull each agent toward the group average (coupling)
        group_average = np.mean(state)
        state = state + coupling * (group_average - state)

        # Record the state
        history[t] = state.copy()

    return history


history = run_simulation(
    n_agents=N_AGENTS,
    n_steps=N_STEPS,
    noise=NOISE,
    coupling=COUPLING,
    corruption_step=CORRUPTION_STEP,
    corruption_magnitude=CORRUPTION_MAGNITUDE,
    corrupted_agent=CORRUPTED_AGENT,
)
print(f'\nSimulation complete. History shape: {history.shape} (steps x agents)')

## Step 3 — Plot the results

Each line is one agent's value over time. The corrupted agent is highlighted in red.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Top panel: all agents over time
for i in range(N_AGENTS):
    color = 'red' if i == CORRUPTED_AGENT else 'steelblue'
    alpha = 1.0 if i == CORRUPTED_AGENT else 0.4
    ax1.plot(history[:, i], color=color, alpha=alpha, linewidth=1.5 if i == CORRUPTED_AGENT else 0.8)

ax1.axvline(x=CORRUPTION_STEP, color='orange', linestyle='--', label='Corruption injected')
ax1.set_title(f'Agent Values Over Time (Coupling = {COUPLING})', fontsize=14)
ax1.set_xlabel('Time Step')
ax1.set_ylabel('Agent Value')
ax1.legend()

# Bottom panel: spread (standard deviation) over time — measure of coherence
spread = np.std(history, axis=1)
ax2.plot(spread, color='teal', linewidth=2)
ax2.axvline(x=CORRUPTION_STEP, color='orange', linestyle='--', label='Corruption injected')
ax2.set_title('Spread (Std Dev) Over Time — Lower = More Coherent', fontsize=14)
ax2.set_xlabel('Time Step')
ax2.set_ylabel('Standard Deviation')
ax2.legend()

plt.tight_layout()
plt.savefig('coupling_simulation.png', dpi=100, bbox_inches='tight')
plt.show()
print('Plot saved as coupling_simulation.png')

## Step 4 — Explore the coupling regimes

Run this cell to compare all four coupling levels on one chart.

In [ ]:
coupling_levels = [0.0, 0.2, 0.5, 0.9]
labels = ['No coupling (colony)', 'Low coupling', 'Medium coupling', 'High coupling (over-coupled)']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (c, label) in enumerate(zip(coupling_levels, labels)):
    h = run_simulation(
        n_agents=N_AGENTS,
        n_steps=N_STEPS,
        noise=NOISE,
        coupling=c,
        corruption_step=CORRUPTION_STEP,
        corruption_magnitude=CORRUPTION_MAGNITUDE,
        corrupted_agent=CORRUPTED_AGENT,
    )
    ax = axes[idx]
    for i in range(N_AGENTS):
        color = 'red' if i == CORRUPTED_AGENT else 'steelblue'
        alpha = 1.0 if i == CORRUPTED_AGENT else 0.35
        ax.plot(h[:, i], color=color, alpha=alpha, linewidth=1.2)
    ax.axvline(x=CORRUPTION_STEP, color='orange', linestyle='--')
    ax.set_title(f'{label}\n(coupling={c})', fontsize=11)
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Value')

plt.suptitle('Four Coupling Regimes — Same Corruption, Different Spread', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('coupling_regimes.png', dpi=100, bbox_inches='tight')
plt.show()
print('Comparison plot saved as coupling_regimes.png')

## Step 5 — Your turn: find the optimal coupling

**TODO:** Complete the cells below.

**Goal:** Find the coupling level that minimizes *post-corruption spread* while keeping *pre-corruption spread* low.

In [ ]:
# TODO: Sweep coupling from 0.0 to 1.0 in steps of 0.05.
# For each coupling level, run the simulation.
# Record:
#   - mean_spread_before: average spread in steps 0 to CORRUPTION_STEP
#   - mean_spread_after: average spread in steps CORRUPTION_STEP to N_STEPS
#
# Then plot both metrics on the same chart with coupling on the x-axis.
# Mark the point where post-corruption spread is minimized.

coupling_sweep = np.arange(0.0, 1.05, 0.05)
before_spreads = []
after_spreads = []

for c in coupling_sweep:
    h = run_simulation(
        n_agents=N_AGENTS,
        n_steps=N_STEPS,
        noise=NOISE,
        coupling=c,
        corruption_step=CORRUPTION_STEP,
        corruption_magnitude=CORRUPTION_MAGNITUDE,
        corrupted_agent=CORRUPTED_AGENT,
    )
    spread = np.std(h, axis=1)
    # TODO: Calculate mean_spread_before and mean_spread_after
    mean_spread_before = None  # your code here
    mean_spread_after = None   # your code here
    before_spreads.append(mean_spread_before)
    after_spreads.append(mean_spread_after)

# TODO: Plot coupling_sweep on x-axis, before_spreads and after_spreads as two lines
# Label your axes and add a legend
print('Sweep complete. Now plot your results.')

In [ ]:
# BONUS TODO: Add error damping.
# Before an agent writes to the shared pool, check if its value is more than
# 2 standard deviations from the current group mean.
# If so, clip it to the mean + 2*std instead of passing the full corrupted value.
#
# Modify the run_simulation function to add this damping option.
# Compare the damped vs. undamped results at coupling = 0.5.
# How much does damping reduce post-corruption spread?

# Your modified simulation function here
pass

## Deliverable

In the cell below, write 3–5 sentences answering:

1. At what coupling level did you observe the best balance of coherence (low pre-corruption spread) and error containment (low post-corruption spread)?
2. What happened at coupling = 0.0? At coupling = 0.9?
3. How does this connect to the biological failure modes described in Chapter 6 — de-integration at low coupling and arrhythmia at high coupling?
4. (If you did the bonus) How much did the damping mechanism help? What is the biological equivalent of the clipping threshold you used?

**Your answer here:** (double-click to edit)

1. 
2. 
3. 
4. 